In [5]:
import arcpy
import os

In [6]:
# ── PATHS ──────────────────────────────────────────────────────────────────────────────────────────────────

# Input LF and HF interpolated backgroud/mission path data CSVs
lf_csv = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\LF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"
hf_csv = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\HF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"


# intermediate polyline feature classes output by 'Coordinate Table to 2-Point Line' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_line_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_line_g4326"
hf_line_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_line_g4326"
# intermediate point feature classes output by 'XY Table to Point' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_points_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_points_g4326"
hf_points_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_points_g4326"


# Output MISSION PATH LINE feature classes (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_line_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_background_line_p26920"
hf_line_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_background_line_p26920"

# Output CALC ATTRIBUTE TABLE point feature classes (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_LF_background_points_p26920"
hf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"

In [7]:
# ── SETTINGS ───────────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True

In [ ]:
# ── 1. CREATE MISSION PATH LINE FROM CSV DATA ────────────────────────────────────────────────────────────

# Coordinate Table To 2-Point Line (Defense Tools): Function
def create_mission_path_line(in_csv, out_line_g4326):
    try:
        arcpy.defense.CoordinateTableTo2PointLine(
            in_table             = in_csv,
            out_feature_class    = out_line_g4326,
            start_x_or_lon_field = "start_longitude_DD",
            end_x_or_lon_field   = "end_longitude_DD",
            # input CSV has coordinates in decimal degrees (DD) with lat and lon stored in * two separate columns [lat] [lon] * (not in one column separated by comma [lat, lon])
            in_coordinate_format = "DD_2",
            start_y_or_lat_field = "start_latitude_DD",
            end_y_or_lat_field   = "end_latitude_DD",
            line_type            = "GEODESIC",
            # output line feature class in EPSG:4326 (native coordinate system of glider data that mission background conditions coordinates are interpolated from)
            coordinate_system    = arcpy.SpatialReference(4326)
        )
        print(f"✅ Created: {out_line_g4326}")
    # print warning message if tool fails
    except Exception as e:
        print(f"\n❌ [FAILURE] Coordinate Table To 2-Point Line failed for {os.path.basename(out_line_g4326)}: {e} \n")

# Call function for LF and HF mission background conditions CSVs
create_mission_path_line(lf_csv, lf_line_g4326)
create_mission_path_line(hf_csv, hf_line_g4326)

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_line_g4326 

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_line_g4326 



In [ ]:
# ── REPROJECT MISSION PATH LINE TO EPSG:26920 ───────────────────────────────────────────────────────────

# Project (Data Management Tools): Function
def reproject_mission_path_line(in_line_g4326, out_line_p26920):
    try:
        arcpy.management.Project(
            in_dataset       = in_line_g4326,
            out_dataset      = out_line_p26920,
            # output coordinate system
            out_coor_system  = arcpy.SpatialReference(26920),
            transform_method = "WGS_1984_(ITRF00)_To_NAD_1983",
            # input coordinate system
            in_coor_system   = arcpy.SpatialReference(4326),
            # * do NOT want 'preserve shape' tool to add extra vertices to lines - want each line to go directly from minute start lat/lon to end lat/lon
            preserve_shape   = "NO_PRESERVE_SHAPE"
        )
        print(f"✅ Reprojected: {os.path.basename(out_line_p26920)}")
    # print warning message if tool fails
    except Exception as e:
        print(f"\n❌ [FAILURE] Reprojection failed for {os.path.basename(out_line_p26920)}: {e} \n")

# Call function for LF and HF mission path polyline feature classes
reproject_mission_path_line(lf_line_g4326, lf_line_p26920)
reproject_mission_path_line(hf_line_g4326, hf_line_p26920)

✅ Reprojected: LF_background_line_p26920
✅ Reprojected: HF_background_line_p26920


In [ ]:
# ── 2. CREATE BACKGROUND CONDITIONS POINT FEATURE CLASS FROM CSV DATA ─────────────────────────────────────

# XY Table To Point (Data Management Tools): Function
def create_background_points(in_csv, out_points_g4326):
    try:
        arcpy.management.XYTableToPoint(
            in_table          = in_csv,
            out_feature_class = out_points_g4326,
            # use lat/lon at * middle-point * of each spectrogram minute 
            x_field           = "av_longitude_DD",
            y_field           = "av_latitude_DD",
            z_field           = None,
            # output point feature class in EPSG:4326 (native coordinate system of glider data that mission background conditions coordinates are interpolated from)
            coordinate_system = arcpy.SpatialReference(4326)
        )
        print(f"✅ Created: {out_points_g4326}")
    # print warning message if tool fails
    except Exception as e:
        print(f"\n❌ [FAILURE] XY Table To Point failed for {os.path.basename(out_points_g4326)}: {e} \n")

# Call function for LF and HF mission background conditions CSVs
create_background_points(lf_csv, lf_points_g4326)
create_background_points(hf_csv, hf_points_g4326)

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_points_g4326 

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_points_g4326 



In [ ]:
# ── REPROJECT MISSION BACKGROUND CONDITIONS POINTS TO EPSG:26920 ─────────────────────────────────────────

# Project (Data Management Tools): Function
def reproject_background_points(in_points_g4326, out_points_p26920):
    try:
        arcpy.management.Project(
            in_dataset       = in_points_g4326,
            out_dataset      = out_points_p26920,
            # output coordinate system
            out_coor_system  = arcpy.SpatialReference(26920),
            transform_method = "WGS_1984_(ITRF00)_To_NAD_1983",
            # input coordinate system
            in_coor_system   = arcpy.SpatialReference(4326),
            # not applicable for point layers (adds extra vertices to lines/polygons)
            preserve_shape   = "NO_PRESERVE_SHAPE"
        )
        print(f"✅ Reprojected: {os.path.basename(out_points_p26920)}")
    # print warning message if tool fails
    except Exception as e:
        print(f"\n❌ [FAILURE] Reprojection failed for {os.path.basename(out_points_p26920)}: {e} \n")

# Call function for LF and HF mission background conditions point feature classes
reproject_background_points(lf_points_g4326, lf_points_p26920)
reproject_background_points(hf_points_g4326, hf_points_p26920)

✅ Reprojected: calc_LF_background_points_p26920
✅ Reprojected: calc_HF_background_points_p26920
